In [10]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import submission_helper

df_train = pd.read_csv("datasets/task3_train.csv")
X_full = df_train.drop(columns=["anomaly"]).values.astype(np.float32)
y_full = df_train["anomaly"].values.astype(np.float32)

X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

class OptimizedAnomalyDetector(nn.Module):
    def __init__(self):
        super(OptimizedAnomalyDetector, self).__init__()
        self.fc1 = nn.Linear(20, 14)
        self.bn1 = nn.BatchNorm1d(14)
        self.fc2 = nn.Linear(14, 7)
        self.bn2 = nn.BatchNorm1d(7)
        self.fc3 = nn.Linear(7, 1)
        self.dropout = nn.Dropout(0.15)
        
    def forward(self, x):
        x = torch.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = torch.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

model = OptimizedAnomalyDetector()
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

X_train_tensor = torch.tensor(X_train)
y_train_tensor = torch.tensor(y_train).reshape(-1, 1)
X_val_tensor = torch.tensor(X_val)
y_val_tensor = torch.tensor(y_val).reshape(-1, 1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

pos_weight = torch.tensor([(len(y_train) - y_train.sum()) / y_train.sum()])
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=0.003, weight_decay=0.005)

best_f1 = 0
best_thresh = 0.5

for epoch in range(250):
    model.train()
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
    
    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val_tensor)
            val_probs = torch.sigmoid(val_outputs).numpy().flatten()
            
            thresholds = np.linspace(0.3, 0.95, 100)
            f1_scores = []
            for thresh in thresholds:
                preds = (val_probs > thresh).astype(int)
                f1_scores.append(f1_score(y_val, preds))
            
            current_f1 = max(f1_scores)
            current_thresh = thresholds[np.argmax(f1_scores)]
            
            if current_f1 > best_f1:
                best_f1 = current_f1
                best_thresh = current_thresh
                torch.save(model.state_dict(), "best_model.pth")

model.load_state_dict(torch.load("best_model.pth"))

model.eval()
with torch.no_grad():
    val_outputs = model(X_val_tensor)
    val_probs = torch.sigmoid(val_outputs).numpy().flatten()
    val_preds = (val_probs > best_thresh).astype(int)
    val_f1 = f1_score(y_val, val_preds)

submission_helper.save_task3_model(model, "task3_model.pkl")

[Task 3] Model saved to 'task3_model.pkl' | Parameters: 449 / 500 (VALID) | Format: TorchScript
